# 03 — Gold/Blue J-Lens × Logit Lens sweep

**Goal.** Apply the fixed base-model J-Lens and vanilla Logit Lens to the
same saved base/Gold/Blue sequences over published standard and direct
prompts.

The long job now runs in small resumable notebook batches. Model/readout code
is visible below, progress is printed by sequence and layer, and every complete
sequence is saved atomically before the next one begins.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT


In [ ]:
RUN_ID = "PASTE_RUN_ID_FROM_NOTEBOOK_01"

from src.experiment_io import open_run
from src.behavior import behavior_dataframe, require_behavior_approval
from src.jlens_sanity import require_sanity_approval

paths, config = open_run(RUN_ID)
require_behavior_approval(paths, config)
require_sanity_approval(paths, config)
behavior = behavior_dataframe(paths)
required_prompts = config["prompts"]["groups"]["lens_sweep"]
required_conditions = config["behavior"]["conditions"]
display(behavior[
    behavior["prompt_id"].isin(required_prompts)
    & behavior["condition"].isin(required_conditions)
][["prompt_id", "prompt_type", "condition", "own_secret_leaked", "generation_token_count"]])


## Measurement plan

- Full layer coverage from the public J-Lens checkpoint.
- Positions: last 10 rendered-input tokens plus every generated token.
- Cheap Gold/Blue candidate logits at every measured layer × position.
- Full-vocabulary ranks/top-10 at every layer for the last input token,
  first generated token, and five generated quantiles.
- Full-vocabulary ranks/top-10 at every position for anchor layers 32, 48, 51.
- Prior confirmatory band: layers 37–58. The full sweep remains exploratory;
  the band summary avoids selecting the best layer after seeing results.
- Outputs containing their own secret remain saved but are excluded from the
  headline metric.

Layer 48 is an a-priori anchor because the pinned Qwen3.6 source reports its
best Activation-Oracle Taboo readout at 75% depth; layer 32 is that source's
mid-stack failure/control, and 51 is a fixed nearby late-stack diagnostic.
This motivates where to inspect, but it does not imply that an AO optimum must
be a J-Lens optimum.

The user-requested restriction to existing prompts means this phase uses the
published standard and direct splits only. Base and the other adapter provide
the main controls; a newly authored unrelated-topic or leakage-positive prompt
would be a separate later condition and is intentionally not introduced here.


## Why this comparison follows the references

The original Taboo work ([2505.14352](https://arxiv.org/abs/2505.14352)) and
the larger secret-elicitation benchmark
([2510.01070](https://arxiv.org/abs/2510.01070)) make Logit Lens the direct
white-box baseline and keep direct black-box attacks behaviorally distinct.
The activation-oracle confidence study
([2605.26045](https://arxiv.org/abs/2605.26045)) shows why scoring a known
candidate set is easier: Gold/Blue candidate metrics are therefore auxiliary,
while full-vocabulary ranks/top-k are required. The natural-censorship study
([2603.05494](https://arxiv.org/abs/2603.05494)) motivates later matched
censorship controls but does not turn this synthetic two-adapter pilot into a
natural-censorship result.


In [ ]:
display(config["readout"])

missing = []
for prompt_id in required_prompts:
    for condition in required_conditions:
        rows = behavior[
            behavior["prompt_id"].eq(prompt_id)
            & behavior["condition"].eq(condition)
        ]
        if len(rows) != 1:
            missing.append((prompt_id, condition, len(rows)))
assert not missing, f"Missing or duplicate behavior rows: {missing}"


## Reuse the loaded model and define the readout math

Notebook 03 now runs the sweep directly in the same kernel. It does not launch
a hidden second process or reload another 27B model. The functions below are
kept in notebook cells so candidate logits, full-vocabulary ranks, layer
transport and position selection can be inspected and edited.


In [ ]:
required_state = [
    "model", "tokenizer", "adapter_names", "token_audit", "lens", "lens_model",
    "runtime", "torch",
]
missing_state = [name for name in required_state if name not in globals()]
if missing_state:
    raise RuntimeError(
        f"Missing in-memory state {missing_state}. Run notebooks 01 and 02 in this kernel."
    )

import os
import time
from jlens.hooks import ActivationRecorder
from src.experiment_io import utc_now

def position_roles(position, prompt_length, sequence_length, input_window):
    roles = []
    if position == prompt_length - 1:
        roles.append("last_input")
    if max(0, prompt_length - input_window) <= position < prompt_length:
        roles.append("last_input_window")
    if position == prompt_length and position < sequence_length:
        roles.append("first_generated")
    if position >= prompt_length:
        roles.append("generated")
    if position == sequence_length - 1 and position >= prompt_length:
        roles.append("last_generated")
    return roles

def generated_quantile_positions(start, stop, quantiles):
    if stop <= start:
        return []
    last = stop - 1
    return sorted({
        min(last, max(start, int(round(start + q * (last - start)))))
        for q in quantiles
    })

# Build a compact output-head slice for Gold/Blue. This avoids a full-vocabulary
# matrix at every token while retaining exact candidate logits everywhere.
candidate_token_ids = sorted({
    token_id
    for audit in token_audit.values()
    for token_id in audit["single_token_ids"]
})
token_id_to_column = {
    token_id: column for column, token_id in enumerate(candidate_token_ids)
}
candidate_columns = {
    word: [token_id_to_column[token_id] for token_id in audit["single_token_ids"]]
    for word, audit in token_audit.items()
}

def selected_candidate_logits(residual):
    # Match the model's final norm and LM head, but multiply only selected rows.
    head_device = lens_model._lm_head.weight.device
    head_dtype = lens_model._lm_head.weight.dtype
    normalized = lens_model._final_norm(
        residual.to(device=head_device, dtype=head_dtype)
    )
    logits = normalized @ lens_model._lm_head.weight[candidate_token_ids].T
    if lens_model._logit_softcap is not None:
        cap = lens_model._logit_softcap
        logits = cap * torch.tanh(logits / cap)
    return logits.float().cpu()

def full_vocabulary_summary(logits):
    candidates = {}
    for word, audit in token_audit.items():
        ids = audit["single_token_ids"]
        scores = logits[ids]
        best_id = int(ids[int(scores.argmax())])
        candidates[word] = {
            "best_token_id": best_id,
            "best_surface_token": tokenizer.decode([best_id]),
            "logit": float(logits[best_id]),
            "rank": int((logits > logits[best_id]).sum()) + 1,
        }
    values, indices = logits.topk(config["readout"]["top_k"])
    top_k = [
        {"token_id": int(token_id), "token": tokenizer.decode([int(token_id)]), "logit": float(value)}
        for value, token_id in zip(values, indices)
    ]
    return {"candidates": candidates, "top_k": top_k}


## One resumable sequence measurement

This is the long model operation, shown in full. It records residuals once,
applies either no transport (Logit Lens) or the fixed Jacobian transport, and
writes a temporary JSONL that is renamed only after the whole sequence passes.
Progress is printed every eight layers.


In [ ]:
def measure_one_sequence(behavior_row, output_path):
    readout = config["readout"]
    prompt_ids = list(behavior_row["prompt_token_ids"])
    generation_ids = list(behavior_row["generation_token_ids"])
    complete_ids = prompt_ids + generation_ids
    assert len(complete_ids) <= runtime["max_sequence_tokens"], (
        len(complete_ids), runtime["max_sequence_tokens"]
    )
    input_ids = torch.tensor([complete_ids], device=lens_model.input_device)
    layers = list(lens.source_layers)
    positions = list(range(max(0, len(prompt_ids) - readout["input_window"]), len(complete_ids)))
    anchor_layers = set(readout["anchor_layers"])

    # Select which positions receive expensive full-vocabulary logits.
    generated_quantiles = set(generated_quantile_positions(
        len(prompt_ids), len(complete_ids), readout["full_vocab_generated_quantiles"]
    ))
    full_positions_by_layer = {}
    for layer in layers:
        if layer in anchor_layers:
            full_positions_by_layer[layer] = set(positions)
        else:
            full_positions_by_layer[layer] = (
                {len(prompt_ids) - 1, len(prompt_ids)} | generated_quantiles
            ) & set(positions)

    condition = behavior_row["condition"]
    if condition == "base":
        model.disable_adapters()
    else:
        model.enable_adapters()
        model.set_adapter(adapter_names[condition])
    try:
        with torch.no_grad(), ActivationRecorder(lens_model.layers, at=layers) as recorder:
            lens_model.forward(input_ids)
    finally:
        model.enable_adapters()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(".jsonl.tmp")
    record_count = 0
    with temporary.open("w", encoding="utf-8") as handle:
        for layer_index, layer in enumerate(layers, start=1):
            source = recorder.activations[layer].detach()[0][positions].float()
            for method in ("logit_lens", "jlens"):
                # This line is the actual distinction between the two methods.
                residual = source if method == "logit_lens" else lens.transport(source, layer)
                chunk_size = readout["position_chunk_size"]
                for chunk_start in range(0, len(positions), chunk_size):
                    chunk_stop = min(len(positions), chunk_start + chunk_size)
                    chunk_positions = positions[chunk_start:chunk_stop]
                    chunk_residual = residual[chunk_start:chunk_stop]
                    selected_logits = selected_candidate_logits(chunk_residual)
                    candidate_scores = {
                        word: selected_logits[:, columns].max(dim=-1).values
                        for word, columns in candidate_columns.items()
                    }
                    full_indices = [
                        index for index, position in enumerate(chunk_positions)
                        if position in full_positions_by_layer[layer]
                    ]
                    full_batch = (
                        lens_model.unembed(chunk_residual[full_indices]).float().cpu()
                        if full_indices else None
                    )
                    full_by_index = {
                        local_index: full_vocabulary_summary(full_batch[batch_index])
                        for batch_index, local_index in enumerate(full_indices)
                    }

                    for local_index, position in enumerate(chunk_positions):
                        scores = {
                            word: float(values[local_index])
                            for word, values in candidate_scores.items()
                        }
                        candidate_ranks = {
                            word: 1 + sum(other > score for other in scores.values())
                            for word, score in scores.items()
                        }
                        record = {
                            "schema_version": 1, "timestamp_utc": utc_now(),
                            "run_id": behavior_row["run_id"],
                            "prompt_id": behavior_row["prompt_id"],
                            "prompt_type": behavior_row["prompt_type"],
                            "split": behavior_row["split"],
                            "condition": condition,
                            "target_word": behavior_row.get("secret"),
                            "source_path": behavior_row["source_path"],
                            "source_line": behavior_row["source_line"],
                            "source_submodule_commit": behavior_row["source_submodule_commit"],
                            "base_model_repo_id": behavior_row["base_model_repo_id"],
                            "base_model_revision": behavior_row["base_model_revision"],
                            "tokenizer_repo_id": behavior_row["tokenizer_repo_id"],
                            "tokenizer_revision": behavior_row["tokenizer_revision"],
                            "adapter_repo_id": behavior_row["adapter_repo_id"],
                            "adapter_revision": behavior_row["adapter_revision"],
                            "jlens_repo_id": behavior_row["jlens_repo_id"],
                            "jlens_revision": behavior_row["jlens_revision"],
                            "jlens_filename": behavior_row["jlens_filename"],
                            "jlens_code_commit": behavior_row["jlens_code_commit"],
                            "runtime_dtype": behavior_row["runtime_dtype"],
                            "attention_implementation": behavior_row["attention_implementation"],
                            "seed": behavior_row["seed"],
                            "output_leaks": behavior_row.get("output_candidate_leaks", []),
                            "own_secret_leaked": behavior_row.get("own_secret_leaked", False),
                            "method": method, "layer": layer, "position": position,
                            "position_roles": position_roles(
                                position, len(prompt_ids), len(complete_ids),
                                readout["input_window"],
                            ),
                            "relative_generated_position": (
                                position - len(prompt_ids)
                                if position >= len(prompt_ids) else None
                            ),
                            "token_id": complete_ids[position],
                            "token": tokenizer.decode([complete_ids[position]]),
                            "candidate_logits": scores,
                            "candidate_ranks": candidate_ranks,
                            "full_vocabulary": full_by_index.get(local_index),
                        }
                        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                        record_count += 1
            if layer_index == 1 or layer_index % 8 == 0 or layer_index == len(layers):
                print(
                    f"  {behavior_row['prompt_id']}/{condition}: layer {layer_index}/{len(layers)}",
                    flush=True,
                )
            del source
            torch.cuda.empty_cache()
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, output_path)
    del recorder
    return record_count


## Run a small visible batch

`SEQUENCES_THIS_RUN = 1` is intentionally conservative: one click performs one
`prompt × condition` sequence. Increase it for an unattended batch. Completed
files are skipped, so rerunning after a disconnect resumes safely.


In [ ]:
SEQUENCES_THIS_RUN = 1  # Set to 30 to finish all currently configured sequences.

selected = behavior[
    behavior["prompt_id"].isin(required_prompts)
    & behavior["condition"].isin(required_conditions)
].sort_values(["prompt_id", "condition"])
cells_dir = paths.lens_dir / "cells"
pending = []
for row in selected.to_dict("records"):
    safe_prompt = "".join(character if character.isalnum() else "_" for character in row["prompt_id"])
    safe_condition = "".join(character if character.isalnum() else "_" for character in row["condition"])
    output = cells_dir / f"{safe_prompt}__{safe_condition}.jsonl"
    if not output.exists():
        pending.append((row, output))

print(f"complete: {len(selected) - len(pending)} / {len(selected)}; pending: {len(pending)}")
for batch_index, (row, output) in enumerate(pending[:SEQUENCES_THIS_RUN], start=1):
    started = time.perf_counter()
    print(f"[{batch_index}/{min(SEQUENCES_THIS_RUN, len(pending))}] start {row['prompt_id']}/{row['condition']}", flush=True)
    records = measure_one_sequence(row, output)
    print(f"saved {output.name}: {records} rows in {time.perf_counter() - started:.1f}s", flush=True)

remaining = len(pending) - min(SEQUENCES_THIS_RUN, len(pending))
print("remaining sequences:", remaining)


In [ ]:
# Export only after every sequence has its completed atomic JSONL file.
cell_files = sorted((paths.lens_dir / "cells").glob("*.jsonl"))
expected_sequences = len(required_prompts) * len(required_conditions)
print(f"completed sequences: {len(cell_files)} / {expected_sequences}")
if len(cell_files) == expected_sequences:
    from src.lens_export_stable import export_lens_parquet
    parquet_path = export_lens_parquet(paths)
    print("Exported:", parquet_path)
else:
    print("Rerun the previous cell until all sequences are complete.")


## Inspect completed export

Run only after the previous cell reports all sequences and writes
`lens_readouts.parquet`.


In [ ]:
parquet_path = paths.result_dir / "lens_readouts.parquet"
assert parquet_path.exists(), "Sweep/export is not complete yet."
sample = pd.read_parquet(parquet_path).head(50)
print("Parquet:", parquet_path, f"{parquet_path.stat().st_size / 2**20:.1f} MiB")
display(sample)
